## Install dependencies

Run the cell below **once** to install the required Python packages. You can skip this step if you already have them installed.

In [ ]:
# Install required packages (run this cell once)
%pip install -q requests
print("✓ Required packages installed.")

# Document Download API - Usage Guide

**Bigdata.com API**

This notebook demonstrates how to download entire documents from the Bigdata.com API using the `GET /documents/<id>` endpoint.

---

## Overview

The Document Download API allows you to retrieve the full content of processed documents in JSON format, including:
- Document metadata (source, URL, etc.)
- Structured content (title, body blocks, tables, lists)
- Entity annotations with positions
- Sentence segmentation

## Response Handling

The API handles documents differently based on size:
- **Small documents (<5MB)**: Returns JSON directly in the response
- **Large documents (≥5MB)**: Returns a pre-signed S3 URL to download the file

The code in this notebook handles both scenarios automatically.


## Setup

### Prerequisites
- Python 3.8+
- `requests` library
- Valid Bigdata.com API key (set as `BIGDATA_API_KEY` environment variable)

You can generate an API Key at [platform.bigdata.com/api-keys](https://platform.bigdata.com/api-keys)


In [ ]:
import os
import json
import requests
from typing import Optional

# Verify API key is available
API_KEY = os.getenv('BIGDATA_API_KEY')
if not API_KEY:
    raise ValueError(
        "BIGDATA_API_KEY not found in environment variables. "
        "Please set it before running this notebook."
    )

print("✓ API key loaded successfully")


## Document Download Function

The function below handles both response scenarios:
1. **Direct JSON response** - Document content returned directly
2. **Pre-signed URL response** - For large files, fetches content from the provided S3 URL


In [ ]:
def download_entire_document(document_id: str) -> dict:
    """
    Downloads an entire document from the Bigdata API.
    
    Handles two response scenarios:
    1. The endpoint returns a JSON directly (documents < 5MB)
    2. The endpoint returns a pre-signed URL that requires a second call 
       to fetch the actual document (documents >= 5MB)
    
    Args:
        document_id: The document ID to download (e.g., '0105A1520E8594CB6B0B8505CB0090AA')
    
    Returns:
        dict: The JSON document data containing:
            - document: metadata (id, source, url)
            - content: title and body blocks with entities
    
    Raises:
        requests.RequestException: If the API request fails
        ValueError: If API key is not configured
    """
    # Get API key from environment
    api_key = os.getenv('BIGDATA_API_KEY')
    if not api_key:
        raise ValueError("BIGDATA_API_KEY not found in environment variables")
    
    # Construct the API URL
    url = f'https://api.bigdata.com/documents/{document_id}'
    
    # Set headers with API key
    headers = {
        'x-api-key': api_key
    }
    
    # Send request to the API
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an exception for bad status codes
    
    # Parse the response
    response_data = response.json()
    
    # Check if response contains a pre-signed URL (large file scenario)
    if 'url' in response_data:
        # File is large, fetch from pre-signed URL
        presigned_url = response_data['url']
        json_response = requests.get(presigned_url)
        json_response.raise_for_status()
        return json_response.json()
    else:
        # JSON is directly in the response
        return response_data

print("✓ Function defined successfully")


## Usage Example

### Download Multiple Documents

Let's download several documents and examine their structure.


In [ ]:
# Sample document IDs to download
DOCUMENT_IDS = [
    "2B18CA1F6E77EF1E7B8284903554A7F6",  ## Cisco to Lead Consortium to Help Retrain Workers
    "1CDF23AE378D92563C2042D578C8415A",  ## Wyebot Announces Integration with Cisco Catalyst Center
    "F12636B27CED06126D46B755FD57F5A6",   ## Kellanova to Close 2 Production Facilities
    "6FA3173E4785CFDF02ACA75AA95D2CEC",   ## https://uk.finance.yahoo.com/news/trump-bought-netflix-warner-bros-132120706.html
    "9AF93B7690059F5D54A25741811FA99C",   ## https://finance.yahoo.com/news/2026-corvette-zr1-first-drive-the-king-of-the-hill-is-back-160040128.html
]



In [ ]:

# Download all documents
documents = {}
for doc_id in DOCUMENT_IDS:
    try:
        print(f"Downloading document: {doc_id}...")
        documents[doc_id] = download_entire_document(doc_id)
        print(f"  ✓ Success")
    except requests.exceptions.HTTPError as e:
        print(f"  ✗ Error: {e.response.status_code} - {e.response.text}")
    except Exception as e:
        print(f"  ✗ Error: {e}")

print(f"\n✓ Downloaded {len(documents)} documents successfully")

### Save Documents to Output Folder

Save all downloaded documents to the `output/` folder with automatic extension detection based on:
1. Original URL file extension (if available in metadata)
2. Default to `.json` for the structured document format


In [ ]:
from pathlib import Path
from urllib.parse import urlparse

def detect_extension_from_url(url: str) -> str:
    """
    Detect file extension from URL.
    
    Args:
        url: The original document URL
        
    Returns:
        str: Detected extension (e.g., '.pdf', '.html') or empty string if not detected
    """
    if not url or url == 'N/A':
        return ''
    
    try:
        parsed = urlparse(url)
        path = parsed.path.lower()
        
        # Common document extensions
        extensions = ['.pdf', '.html', '.htm', '.doc', '.docx', '.txt', '.xml', '.xlsx', '.csv']
        for ext in extensions:
            if path.endswith(ext):
                return ext
    except:
        pass
    
    return ''


def save_document(doc_id: str, doc: dict, output_dir: str = "output") -> str:
    """
    Save a downloaded document to the output folder.
    
    Args:
        doc_id: The document ID (used as filename)
        doc: The document dictionary
        output_dir: Output directory path
        
    Returns:
        str: Path to the saved file
    """
    # Create output directory if it doesn't exist
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Try to detect extension from original URL
    url = doc.get('document', {}).get('metadata', {}).get('url', '')
    original_ext = detect_extension_from_url(url)
    
    # Use .json as default since we're saving structured JSON content
    # Add original extension as suffix for reference if detected
    if original_ext:
        filename = f"{doc_id}_original{original_ext}.json"
    else:
        filename = f"{doc_id}.json"
    
    file_path = output_path / filename
    
    # Save document as JSON
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, indent=2, ensure_ascii=False)
    
    return str(file_path)


# Save all downloaded documents
OUTPUT_DIR = "output"
print(f"Saving documents to '{OUTPUT_DIR}/' folder...")
print("-" * 60)

saved_files = []
for doc_id, doc in documents.items():
    try:
        file_path = save_document(doc_id, doc, OUTPUT_DIR)
        file_size = os.path.getsize(file_path)
        saved_files.append((doc_id, file_path, file_size))
        print(f"✓ {doc_id} -> {file_path} ({file_size:,} bytes)")
    except Exception as e:
        print(f"✗ {doc_id} -> Error: {e}")

print("-" * 60)
print(f"✓ Saved {len(saved_files)} documents to '{OUTPUT_DIR}/' folder")


### Extract Full Text Content

Concatenate all body text blocks into a single readable text.



In [ ]:
def extract_body_text(doc: dict) -> str:
    """
    Extract and concatenate all body text from a document.
    
    Args:
        doc: The document dictionary
        
    Returns:
        str: Concatenated body text with paragraph breaks
    """
    body = doc.get('content', {}).get('body', [])
    texts = [block.get('text', '') for block in body if block.get('text')]
    return '\n\n'.join(texts)


# Extract and print full text from the first document
if documents:
    first_doc_id = list(documents.keys())[0]
    doc = documents[first_doc_id]
    
    title = doc.get('content', {}).get('title', {}).get('text', 'N/A')
    body_text = extract_body_text(doc)
    
    print("=" * 80)
    print(f"DOCUMENT: {first_doc_id}")
    print("=" * 80)
    print(f"\n📌 TITLE: {title}\n")
    print("-" * 80)
    print("FULL BODY TEXT:")
    print("-" * 80)
    print(body_text)
else:
    print("No documents available.")


In [ ]:
# Save title and body text to a file for each document (filename: {document_id}.txt)
OUTPUT_DIR_TXT = Path(OUTPUT_DIR)

for doc_id, doc in documents.items():
    title = doc.get('content', {}).get('title', {}).get('text', 'N/A')
    body_text = extract_body_text(doc)
    file_path = OUTPUT_DIR_TXT / f"{doc_id}.txt"
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(f"{title}\n\n")
        f.write(body_text)
    print(f"✓ {doc_id} -> {file_path}")

### Summary of All Downloaded Documents

Display a summary table of all downloaded documents.


In [ ]:
# Display summary of all downloaded documents
print("=" * 100)
print("DOWNLOADED DOCUMENTS SUMMARY")
print("=" * 100)
print(f"\n{'Document ID':<36} {'Source':<25} {'Title (truncated)':<35}")
print("-" * 100)

for doc_id, doc in documents.items():
    source_name = doc.get('document', {}).get('source', {}).get('name', 'N/A')[:24]
    title = doc.get('content', {}).get('title', {}).get('text', 'N/A')[:34]
    url = doc.get('document', {}).get('metadata', {}).get('url', 'N/A')
    print(f"{doc_id:<36} {source_name:<25} {title:<35} {url}")
